In [11]:
import os
import sys
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import LambdaLR

torch.manual_seed(8008135)

NOTEBOOK_DIR = Path.cwd()
CODE_DIR = NOTEBOOK_DIR.parent

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

print("CODE_DIR:", CODE_DIR)
print("CODE_DIR contents:", os.listdir(CODE_DIR))

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print(f"Device set to {device}")

if device.type == "cuda":
    torch.set_float32_matmul_precision("high")

CODE_DIR: /home/daniel/HRM_Reconstruction/code
CODE_DIR contents: ['GPT2_Model', 'Utils', 'HRM_Model', 'Datasets', '.DS_Store', 'BiLSTM_Model', 'Sudoku', 'BERT_Model']
Device set to cuda


In [12]:
from Sudoku.sudoku import print_sudoku_comparison

from Datasets.Sudoku_DataLoader import get_loaders

from HRM_Model.HRM_Model import HRM
from HRM_Model.HRM_Components import Encoder, HighLevel, LowLevel, Head
from HRM_Model.HRM_Train import train_hrm_deepsup

from Utils.schedules import cosine_schedule_with_warmup_lr_lambda
from Utils.checkpointing import load_checkpoint
from Utils.visualization import show_sudoku_predictions

In [ ]:
train_size = 2**18
test_size = 2**15
batch_size = 2**7

train_dataloader, val_dataloader = get_loaders(
    train_size=train_size,
    test_size=test_size,
    batch_size=batch_size,
)

Map: 100%|██████████| 256/256 [00:00<00:00, 28120.20 examples/s]


In [14]:
# Model hyperparameters
d_model = 512
M = 8
N = 2
T = 2

n_layers = 4
n_heads = 8
vocab_size = 10
dropout = 0.2

# Training hyperparameters
lr = 1e-4
beta1 = 0.9
beta2 = 0.95
weight_decay = 0.1
num_epochs = 20

checkpoint_dir = "checkpoints"

In [15]:
high_level = HighLevel(
    d_model=d_model,
    n_layers=n_layers,
    n_heads=n_heads,
    intermediate_size=4 * d_model,
    dropout=dropout,
)

low_level = LowLevel(
    d_model=d_model,
    n_layers=n_layers,
    n_heads=n_heads,
    intermediate_size=4 * d_model,
    dropout=dropout,
)

encoder = Encoder(
    vocab_size=vocab_size,
    d_model=d_model,
)

head = Head(
    d_model=d_model,
    vocab_size=vocab_size,
)

HRM_model = HRM(
    L_module=low_level,
    H_module=high_level,
    encoder=encoder,
    head=head,
    M=M,
    N=N,
    T=T,
    max_len=81,
    d_model=d_model,
).to(device)

print(
    "Number of trainable parameters:",
    f"{sum(p.numel() for p in HRM_model.parameters() if p.requires_grad):,}",
)

Number of trainable parameters: 33,572,864


In [16]:
optimizer = optim.AdamW(
    HRM_model.parameters(),
    lr=lr,
    betas=(beta1, beta2),
    weight_decay=weight_decay,
)

num_training_steps = len(train_dataloader) * num_epochs * M
num_warmup_steps = int(0.05 * num_training_steps)

scheduler = LambdaLR(
    optimizer,
    lr_lambda=lambda step: cosine_schedule_with_warmup_lr_lambda(
        step,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=num_training_steps,
        min_ratio=0.1,
    ),
)

print("num_training_steps:", num_training_steps)
print("num_warmup_steps:", num_warmup_steps)

num_training_steps: 2560
num_warmup_steps: 128


In [18]:
HRM_model, best_metric, history = train_hrm_deepsup(
    model=HRM_model,
    train_loader=train_dataloader,
    optimizer=optimizer,
    loss_fn=nn.CrossEntropyLoss(ignore_index=-100),
    device=device,
    scheduler=scheduler,
    num_epochs=num_epochs,
    checkpoint_dir=checkpoint_dir,
    checkpoint_every=5,
    validate_every=5,
    val_loader=val_dataloader,
    step_val_every=8,
    step_val_batches=1,
)

HRM_model.eval()

print("Best board accuracy used for checkpointing:", best_metric)

Number of trainable parameters: 33,572,864


Epoch 1: 100%|██████████| 16/16 [00:16<00:00,  1.05s/it]


Epoch 1: Avg Train Loss = 2.0501, Train Token Accuracy = 18.49%, Train Board Accuracy = 0.00%, LR = 9.93e-05


Epoch 2: 100%|██████████| 16/16 [00:16<00:00,  1.05s/it]


Epoch 2: Avg Train Loss = 1.9517, Train Token Accuracy = 22.84%, Train Board Accuracy = 0.00%, LR = 9.75e-05


...

Val Loss = 3.9354, Val Token Accuracy = 22.69%, Val Board Accuracy = 0.00%



Epoch 11: 100%|██████████| 16/16 [00:16<00:00,  1.06s/it]


Epoch 11: Avg Train Loss = 0.4145, Train Token Accuracy = 92.80%, Train Board Accuracy = 5.96%, LR = 4.37e-05


Epoch 12:  12%|█▎        | 2/16 [00:02<00:18,  1.34s/it]


KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt

train_steps = history["step"]
train_loss = history["train_loss"]

val_steps = [
    s for s, v in zip(history["step"], history["val_loss"])
    if v is not None
]

val_loss = [
    v for v in history["val_loss"]
    if v is not None
]

plt.figure(figsize=(10, 5))
plt.plot(train_steps, train_loss, label="Train loss", linewidth=1)

if len(val_loss) > 0:
    plt.plot(val_steps, val_loss, label="Validation (subset)", linewidth=2)

plt.xlabel("Training step")
plt.ylabel("Loss")
plt.title("HRM Loss vs Training Step")
plt.legend()
plt.grid(True)
plt.show()

NameError: name 'history' is not defined

In [ ]:
show_sudoku_predictions(
    HRM_model,
    val_dataloader,
    device,
    print_sudoku_comparison,
    num_examples=10,
)

/tmp/ipykernel_27927/1568551438.py:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.long)


Input / Prediction
5 9 8 | 4 6 1 | 2 7 3 
7 3 1 | 5 2 8 | 4 9 6 
2 6 4 | 3 9 7 | 5 8 1 
---------------------------------
8 7 9 | 6 3 2 | 1 4 5 
4 2 6 | 1 5 9 | 7 3 8 
1 5 3 | 8 7 4 | 9 6 2 
---------------------------------
6 1 5 | 9 4 3 | 8 2 7 
3 4 2 | 7 8 5 | 6 1 9 
9 8 7 | 2 1 6 | 3 5 4 

Token Accuracy: 1.0
Input / Prediction
2 9 7 | 4 5 6 | 1 8 3 
3 4 5 | 2 8 1 | 9 6 7 
8 6 1 | 7 9 3 | 2 4 5 
---------------------------------
9 3 4 | 6 2 5 | 7 1 8 
5 1 2 | 8 7 4 | 6 3 9 
6 7 8 | 1 3 9 | 4 5 2 
---------------------------------
7 8 6 | 5 1 2 | 3 9 4 
1 5 9 | 3 4 7 | 8 2 6 
4 2 3 | 9 6 8 | 5 7 1 

Token Accuracy: 1.0
Input / Prediction
5 2 9 | 7 6 8 | 3 4 1 
7 6 3 | 9 1 4 | 2 8 5 
1 4 8 | 5 2 3 | 9 6 7 
---------------------------------
4 3 5 | 6 7 1 | 8 2 9 
8 9 2 | 3 4 5 | 7 1 6 
6 1 7 | 8 9 2 | 4 5 3 
---------------------------------
9 8 4 | 1 3 6 | 5 7 2 
2 7 6 | 4 5 9 | 1 3 8 
3 5 1 | 2 8 7 | 6 9 4 

Token Accuracy: 1.0
Input / Prediction
5 6 1 | 7 3 2 | 4 8 9 
2 8 4 | 1 5 9

In [ ]:
torch.save(
    HRM_model.state_dict(),
    "checkpoints/hrm_final_state_dict.pt",
)